# B1 - Variational Autoencoder (VAE) on MNIST
a. The encoder ends in **two separate linear layers**: $\mu$ and $\log\sigma^2$
b. **Reparameterization trick**: $z = \mu + e^{0.5\log\sigma^2}\odot\epsilon$
c. **Decoder** plus **total loss** = BCE reconstruction + KL divergence

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
# Load + preprocess
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
train_ds = tf.data.Dataset.from_tensor_slices(x_train).shuffle(10000).batch(128)
print(x_train.shape, x_test.shape)

In [ ]:
LATENT_DIM = 2

class VAE(keras.Model):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        # Encoder
        self.flatten = layers.Flatten()
        self.enc1 = layers.Dense(512, activation='relu')
        self.enc2 = layers.Dense(256, activation='relu')
        self.z_mean = layers.Dense(latent_dim, name='mu')         # linear output 1: mean
        self.z_log_var = layers.Dense(latent_dim, name='log_var') # linear output 2: log variance
        # Decoder
        self.dec1 = layers.Dense(256, activation='relu')
        self.dec2 = layers.Dense(512, activation='relu')
        self.dec_out = layers.Dense(784, activation='sigmoid')
        self.reshape = layers.Reshape((28, 28))

    def encode(self, x):
        h = self.enc2(self.enc1(self.flatten(x)))
        return self.z_mean(h), self.z_log_var(h)

    def reparameterize(self, mu, log_var):
        eps = tf.random.normal(tf.shape(mu))
        return mu + tf.exp(0.5 * log_var) * eps

    def decode(self, z):
        return self.reshape(self.dec_out(self.dec2(self.dec1(z))))

    def call(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var


def vae_loss(x, x_hat, mu, log_var):
    x_hat = tf.clip_by_value(x_hat, 1e-7, 1 - 1e-7)
    recon = -tf.reduce_sum(x * tf.math.log(x_hat) + (1 - x) * tf.math.log(1 - x_hat), axis=[1, 2])
    kl = -0.5 * tf.reduce_sum(1 + log_var - tf.square(mu) - tf.exp(log_var), axis=1)
    return tf.reduce_mean(recon + kl), tf.reduce_mean(recon), tf.reduce_mean(kl)

In [ ]:
vae = VAE()
optimizer = keras.optimizers.Adam(1e-3)

@tf.function
def train_step(x):
    with tf.GradientTape() as tape:
        x_hat, mu, log_var = vae(x)
        loss, recon, kl = vae_loss(x, x_hat, mu, log_var)
    grads = tape.gradient(loss, vae.trainable_variables)
    optimizer.apply_gradients(zip(grads, vae.trainable_variables))
    return loss, recon, kl

EPOCHS = 10
history = []
for epoch in range(EPOCHS):
    losses = [train_step(x) for x in train_ds]
    loss, recon, kl = np.mean(np.array(losses), axis=0)
    history.append(loss)
    print(f"Epoch {epoch+1}: total={loss:.2f}  recon={recon:.2f}  KL={kl:.2f}")

In [ ]:
plt.plot(history); plt.xlabel('epoch'); plt.ylabel('total loss'); plt.title('VAE training loss'); plt.show()

# Test-set evaluation
x_hat, mu, log_var = vae(x_test)
loss, recon, kl = vae_loss(x_test, x_hat, mu, log_var)
print(f"Test: total={loss:.2f}  recon={recon:.2f}  KL={kl:.2f}")

In [ ]:
# Original vs reconstruction
n = 10
plt.figure(figsize=(12, 3))
for i in range(n):
    plt.subplot(2, n, i + 1); plt.imshow(x_test[i], cmap='gray'); plt.axis('off')
    plt.subplot(2, n, n + i + 1); plt.imshow(x_hat[i], cmap='gray'); plt.axis('off')
plt.suptitle('Top: original   Bottom: reconstruction'); plt.show()

# New digits from z ~ N(0, I)
gen = vae.decode(tf.random.normal((16, LATENT_DIM))).numpy()
plt.figure(figsize=(4, 4))
for i in range(16):
    plt.subplot(4, 4, i + 1); plt.imshow(gen[i], cmap='gray'); plt.axis('off')
plt.suptitle('Generated digits'); plt.show()

# 2-D latent space
plt.figure(figsize=(6, 5))
plt.scatter(mu[:, 0], mu[:, 1], c=y_test, cmap='tab10', s=2); plt.colorbar(); plt.title('Latent space (mu)'); plt.show()